# 🧪 NotebookMind — Test Plan

This notebook has two parts:

1. **Automated backend checks** (the code cells below) — run them top-to-bottom. They hit the live Supabase project and verify the accounts, roles, enrollment, the opt-in course-scoped leaderboard, and that migration 3 is applied. Each check prints ✅ / ❌.
2. **Manual UI checklist** (at the bottom) — things to click through inside the NotebookMind panel in JupyterLab.

> Requires internet + the Supabase project to be **un-paused**. The anon key and test credentials below are non-secret (publishable key + test accounts).

In [ ]:
import os, json, urllib.request, urllib.error

# Falls back to the known project if env vars are not set (start.ps1 sets these).
SUPABASE_URL = os.environ.get('SUPABASE_URL', 'https://wzooaiwmnsqvxxcoormp.supabase.co')
ANON_KEY     = os.environ.get('SUPABASE_ANON_KEY', 'sb_publishable_HcZ7s3WcO2HJD84XHVS3Yg_ZP19IhfR')
DEMO_COURSE_ID = '00000000-0000-0000-0000-000000000001'

TEACHER = ('notebookmind.prof@gmail.com', 'Teacher123!')
STUDENT = ('notebookmind.student@gmail.com', 'Student123!')

print('Target:', SUPABASE_URL)

In [ ]:
RESULTS = []

def check(name, ok, detail=''):
    status = 'PASS' if ok else 'FAIL'
    RESULTS.append((status, name, detail))
    print(('\u2705' if ok else '\u274c'), name, ('\u2014 ' + str(detail)) if detail else '')
    return ok

def http(method, path, token=None, body=None):
    url = path if path.startswith('http') else SUPABASE_URL + path
    headers = {'apikey': ANON_KEY, 'Content-Type': 'application/json'}
    if token:
        headers['Authorization'] = 'Bearer ' + token
    data = json.dumps(body).encode() if body is not None else None
    req = urllib.request.Request(url, data=data, headers=headers, method=method)
    try:
        with urllib.request.urlopen(req, timeout=20) as r:
            txt = r.read().decode()
            return r.status, (json.loads(txt) if txt else None)
    except urllib.error.HTTPError as e:
        txt = e.read().decode()
        try:
            return e.code, json.loads(txt)
        except Exception:
            return e.code, txt
    except Exception as e:
        return 0, str(e)

def sign_in(email, password):
    st, body = http('POST', '/auth/v1/token?grant_type=password', body={'email': email, 'password': password})
    if st == 200 and isinstance(body, dict):
        return body['access_token'], body['user']['id']
    return None, None

print('helpers ready')

## 1 — Connectivity & authentication

In [ ]:
st, _ = http('GET', '/auth/v1/health')
check('Supabase project reachable (not paused)', st == 200, 'status ' + str(st))

t_tok, t_uid = sign_in(*TEACHER)
check('Professor can sign in', t_tok is not None, TEACHER[0])

s_tok, s_uid = sign_in(*STUDENT)
check('Student can sign in', s_tok is not None, STUDENT[0])

## 2 — Roles & course wiring

In [ ]:
st, prof = http('GET', '/rest/v1/profiles?select=role,display_name&user_id=eq.' + (t_uid or ''), token=t_tok)
ok = isinstance(prof, list) and prof and prof[0].get('role') == 'teacher'
check('Professor profile role = teacher', ok, str(prof[0]) if isinstance(prof, list) and prof else str(prof))

st, stud = http('GET', '/rest/v1/profiles?select=role,display_name&user_id=eq.' + (s_uid or ''), token=s_tok)
ok = isinstance(stud, list) and stud and stud[0].get('role') == 'student'
check('Student profile role = student', ok, str(stud[0]) if isinstance(stud, list) and stud else str(stud))

st, course = http('GET', '/rest/v1/courses?select=name,teacher_id&id=eq.' + DEMO_COURSE_ID, token=t_tok)
ok = isinstance(course, list) and course and course[0].get('teacher_id') == t_uid
check('Professor owns the demo course', ok, str(course))

st, enr = http('GET', '/rest/v1/course_enrollments?select=course_id&course_id=eq.' + DEMO_COURSE_ID, token=s_tok)
check('Student is enrolled in the demo course', isinstance(enr, list) and len(enr) >= 1, str(enr))

## 3 — Opt-in, course-scoped leaderboard (migration 3)

Verifies `profiles.leaderboard_opt_in` and the `get_course_leaderboard` RPC: an opted-out student is hidden, an opted-in student appears. Leaves the student **opted-out** at the end so the opt-in card shows when you test the UI.

In [ ]:
# Start opted-out
http('PATCH', '/rest/v1/profiles?user_id=eq.' + (s_uid or ''), token=s_tok, body={'leaderboard_opt_in': False})

st, lb0 = http('POST', '/rest/v1/rpc/get_course_leaderboard', token=s_tok, body={'p_course_id': DEMO_COURSE_ID})
check('Leaderboard RPC exists & callable by enrolled student', st == 200, 'status ' + str(st))
names0 = [r.get('display_name') for r in lb0] if isinstance(lb0, list) else []
check('Opted-out student is NOT on the leaderboard', 'Test Student' not in names0, str(names0))

# Opt in
http('PATCH', '/rest/v1/profiles?user_id=eq.' + (s_uid or ''), token=s_tok, body={'leaderboard_opt_in': True})
st, lb1 = http('POST', '/rest/v1/rpc/get_course_leaderboard', token=s_tok, body={'p_course_id': DEMO_COURSE_ID})
names1 = [r.get('display_name') for r in lb1] if isinstance(lb1, list) else []
check('Opted-in student APPEARS on the leaderboard', 'Test Student' in names1, str(names1))

# Reset so the opt-in card shows in the UI
http('PATCH', '/rest/v1/profiles?user_id=eq.' + (s_uid or ''), token=s_tok, body={'leaderboard_opt_in': False})
print('reset student to opted-out for manual UI test')

## 4 — Course content tables reachable

In [ ]:
st, weeks = http('GET', '/rest/v1/course_weeks?select=week_number,theme&course_id=eq.' + DEMO_COURSE_ID, token=s_tok)
n = len(weeks) if isinstance(weeks, list) else weeks
check('course_weeks readable', st == 200, 'status ' + str(st) + ', rows=' + str(n))

st, docs = http('GET', '/rest/v1/documents?select=id,title&limit=1', token=s_tok)
check('documents table readable', st == 200, 'status ' + str(st))

st, fc = http('GET', '/rest/v1/flashcards?select=id&limit=1', token=s_tok)
check('flashcards table readable', st == 200, 'status ' + str(st))

## Summary

In [ ]:
passed = sum(1 for r in RESULTS if r[0] == 'PASS')
print('=' * 44)
print(str(passed) + '/' + str(len(RESULTS)) + ' checks passed')
for s, n, d in RESULTS:
    if s == 'FAIL':
        print('  \u274c', n, '\u2014', d)
if passed == len(RESULTS):
    print('\U0001f389 All backend checks passed.')

---
## 🖱️ Manual UI checklist (inside JupyterLab)

Reload the JupyterLab tab, then click the orange **\U0001f4d3 NotebookMind** button in the top bar.

### A. Auth
- [ ] Login screen shows **no** "No Supabase connection" banner (backend is connected)
- [ ] Sign in as **professor** — `notebookmind.prof@gmail.com` / `Teacher123!`
- [ ] Sign out, sign in as **student** — `notebookmind.student@gmail.com` / `Student123!`

### B. Explain mode — inline slides (the new feature)
- [ ] Open a notebook → **Explain**
- [ ] Select a cell → **AI tab** shows an embedded **slide miniature** in the cell context
- [ ] **View full deck →** opens the full slide deck
- [ ] On a cell with no linked slide: **\U0001f64b Request slides** shows a confirmation
- [ ] **\u26a0\ufe0f Mark missing info** shows a confirmation

### C. Learn mode
- [ ] Challenges render (multiple choice / find the bug / fill in)
- [ ] Code runs against the Python kernel and output is compared
- [ ] XP increases on success

### D. Leaderboard — opt-in + course-scoped (the new behaviour)
- [ ] As **student**, open **Leaderboard** → you see the **opt-in card** (not a table)
- [ ] Click **Show me on the leaderboard** → the course leaderboard table appears
- [ ] You appear in the list; **Hide me from the leaderboard** returns to the opt-in card

### E. Slides & Papers reader
- [ ] Upload a PDF → sections + pages render
- [ ] Add a section note (auto-saves)
- [ ] Generate a quiz / flashcards (needs the Gemini key, which start.ps1 sets)
- [ ] **Reload** JupyterLab → notes & flashcards **persist** (saved to Supabase)

### F. Teacher dashboard
- [ ] Sign in as **professor** → open **Teacher**
- [ ] The **anonymous aggregate** view loads (cell failure stats, **no** per-student names)
